In [1]:
import pandas as pd

datos = pd.read_csv('../../data/bronze/staging/stg1_casas_gam_full2.csv')
datos.info()
datos.drop(columns=['FlagsFeatures','IsAuction', 'Address'], inplace=True)
datos[datos['Total_area_m2'] < 10][['Total_area_m2','Built_area_m2']].head(100)

<class 'pandas.DataFrame'>
RangeIndex: 546 entries, 0 to 545
Data columns (total 17 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Total_area_m2     546 non-null    int64  
 1   Built_area_m2     546 non-null    int64  
 2   Rooms             546 non-null    int64  
 3   Bathrooms         546 non-null    int64  
 4   HalfBathrooms     546 non-null    int64  
 5   Parkings          546 non-null    int64  
 6   Antiguedad        504 non-null    float64
 7   Colonia           546 non-null    str    
 8   Price             546 non-null    int64  
 9   Maintenance_cost  546 non-null    int64  
 10  Address           543 non-null    str    
 11  PropertyType      546 non-null    str    
 12  IsAuction         546 non-null    bool   
 13  FlagsFeatures     56 non-null     str    
 14  HaveVideo         546 non-null    bool   
 15  Latitude          546 non-null    float64
 16  Longitude         546 non-null    float64
dtypes: bool(

,Total_area_m2,Built_area_m2
85,1,54
245,0,0
357,0,58
397,1,65
398,0,64
468,0,50
471,0,58
491,0,75
512,0,300
532,0,117


In [2]:
datos.loc[datos['Total_area_m2'] < 10, 'Total_area_m2'] = datos['Built_area_m2']
print(datos[datos['Total_area_m2'] < 10][['Total_area_m2', 'Built_area_m2']].head())

# We delete the values where both areas are 0
datos = datos[~((datos['Total_area_m2'] == 0) & (datos['Built_area_m2'] == 0))]


print(f"Total registers on the DataFrame After the dropping: {len(datos)}")


     Total_area_m2  Built_area_m2
245              0              0
Total registers on the DataFrame After the dropping: 545


In [3]:
# Aplicar la asignación condicional usando .loc[filas, columna]
datos.loc[datos['Built_area_m2'] < 10, 'Built_area_m2'] = datos['Total_area_m2']

# (Opcional) Verificar que ya no existan registros menores a 10 m2 en 'Built_area_m2'
# (a menos que 'Total_area_m2' también haya sido menor a 10)
print("Registros restantes con Built_area_m2 < 10:")
print(len(datos[datos['Built_area_m2'] < 10]))


Registros restantes con Built_area_m2 < 10:
0


In [4]:
datos = datos[(datos['Rooms'] <= 9)]
datos = datos[(datos['Bathrooms'] >= 1)]
datos = datos[datos['Parkings'] <= 12]
# Conservar solo los datos que están por debajo del percentil 99.5
datos = datos[datos['Price'] < datos['Price'].quantile(.995)]
datos['PropertyType'] = datos['PropertyType'].replace('Casa en condominio', 'Casa')


In [5]:
mapeo_colonias = {
    'Gertrudiz Sánchez': 'Gertrudis Sánchez',
    'Gabriel HernAndez': 'Gabriel Hernández',
    'Ampliación Gabriel HernAndez': 'Ampliación Gabriel Hernández',
    'La Purísima TicomAn': 'La Purísima Ticomán',
    'Barrio Candelaria TicomAn': 'Barrio Candelaria Ticomán',
    'Santa Maria Ticoman': 'Santa María Ticomán',
    'Ticoman': 'Ticomán',
    'Maximino Avila Camacho': 'Maximino Ávila Camacho',
    'Nueva Tenochtitlan': 'Nueva Tenochtitlán',
    'General Felipe Berriozabal': 'General Felipe Berriozábal',
    'Constitución de La República': 'Constitución de la República',
    'Defensores de La República': 'Defensores de la República',
    'San José de La Escalera': 'San José de la Escalera',
    'San Pedro El Chico': 'San Pedro el Chico'
}

colonia_original = datos['Colonia'].copy()
datos['Colonia'] = datos['Colonia'].replace(mapeo_colonias)

cambios_colonia = pd.DataFrame({
    'Colonia original': colonia_original,
    'Colonia estandarizada': datos['Colonia']
})
cambios_colonia = cambios_colonia[colonia_original != datos['Colonia']]
display(
    cambios_colonia.groupby(
        ['Colonia original', 'Colonia estandarizada']
    ).size().reset_index(name='Registros')
)

print(f"Colonias únicas después de estandarizar: {datos['Colonia'].nunique()}")
datos['Colonia'].unique()

,Colonia original,Colonia estandarizada,Registros
0,Ampliación Gabriel HernAndez,Ampliación Gabriel Hernández,2
1,Barrio Candelaria TicomAn,Barrio Candelaria Ticomán,1
2,Constitución de La República,Constitución de la República,7
3,Defensores de La República,Defensores de la República,1
4,Gabriel HernAndez,Gabriel Hernández,1
5,General Felipe Berriozabal,General Felipe Berriozábal,1
6,Gertrudiz Sánchez,Gertrudis Sánchez,7
7,La Purísima TicomAn,La Purísima Ticomán,5
8,Maximino Avila Camacho,Maximino Ávila Camacho,1
9,Nueva Tenochtitlan,Nueva Tenochtitlán,1


Colonias únicas después de estandarizar: 99


<StringArray>
[                 'Torres Lindavista',                       'Panamericana',
                   'Granjas Modernas',                  'Guadalupe Tepeyac',
                   'Lindavista Norte',                         'Lindavista',
                     'Lindavista Sur',                    'Pradera II Secc',
                'Santa María Ticomán',                     'Ferrocarrilera',
                           'El Coyol',                         'Industrial',
          'Barrio Candelaria Ticomán',               'Guadalupe Proletaria',
            'Villa Gustavo A. Madero',             'Ampliación Providencia',
                       '15 de Agosto',                 'San Juan de Aragón',
                            'Vallejo',                  'Gustavo A. Madero',
           'Magdalena de las Salinas',                   'Nueva Atzacoalco',
                        'Providencia',              'Planetario Lindavista',
                        'DM Nacional',           'Nueva Indust

In [6]:
import unicodedata

def quitar_acentos(texto):
    return ''.join(
        caracter
        for caracter in unicodedata.normalize('NFKD', texto)
        if not unicodedata.combining(caracter)
    )

datos['Colonia'] = (
    datos['Colonia']
    .map(quitar_acentos)
    .str.lower()
    .str.replace(r'\s+', '_', regex=True)
    .str.strip('_')
)

media_antiguedad_colonia = datos.groupby('Colonia')['Antiguedad'].transform('mean')
datos['Antiguedad'] = datos['Antiguedad'].fillna(media_antiguedad_colonia)

datos['Colonia'].unique()

<StringArray>
[                 'torres_lindavista',                       'panamericana',
                   'granjas_modernas',                  'guadalupe_tepeyac',
                   'lindavista_norte',                         'lindavista',
                     'lindavista_sur',                    'pradera_ii_secc',
                'santa_maria_ticoman',                     'ferrocarrilera',
                           'el_coyol',                         'industrial',
          'barrio_candelaria_ticoman',               'guadalupe_proletaria',
            'villa_gustavo_a._madero',             'ampliacion_providencia',
                       '15_de_agosto',                 'san_juan_de_aragon',
                            'vallejo',                  'gustavo_a._madero',
           'magdalena_de_las_salinas',                   'nueva_atzacoalco',
                        'providencia',              'planetario_lindavista',
                        'dm_nacional',           'nueva_indust

In [7]:
datos.drop(columns=['HaveVideo'], inplace=True)

In [8]:
datos.drop(columns=['Antiguedad'], inplace=True)

In [9]:
datos.to_csv('../../data/silver/cleaned_houses_inmuebles_v3.csv', index=False)